In [ ]:
# Dependencies

import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import IsolationForest, RandomForestClassifier
from sklearn.svm import OneClassSVM
from sklearn.neighbors import LocalOutlierFactor
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from sklearn.impute import SimpleImputer
from sklearn.metrics import (classification_report, confusion_matrix, roc_auc_score, 
                             precision_recall_curve, roc_curve, f1_score, precision_score, recall_score)
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import re
import os
warnings.filterwarnings('ignore')

## Model Definition

### Defining function for Model used, Train Model, and Cross Validations

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.svm import SVC

def get_all_models(n_features):
    """
    Return dictionaries of UNFITTED models for:
    - supervised_smote      (no class_weight, oversampling used in CV)
    - supervised_weighted   (class_weight='balanced', no oversampling)
    """

    models = {
        # =========================================================
        # 1) MODELS FOR SMOTE/ADASYN PIPELINE
        # =========================================================
        'supervised_smote': {
            'Random Forest (SMOTE)': RandomForestClassifier(
                n_estimators=200,
                random_state=42,
                max_depth=n_features,
                n_jobs=-1
            ),

            'XGBoost (SMOTE)': XGBClassifier(
                n_estimators=200,
                random_state=42,
                max_depth=n_features,
                eval_metric='logloss'
            ),

            'Logistic Regression (SMOTE)': LogisticRegression(
                C=0.1,
                random_state=42,
                max_iter=1000,
                solver='liblinear'
            ),

            'Decision Tree (SMOTE)': DecisionTreeClassifier(
                random_state=42,
                max_depth=n_features,
                max_features=n_features,
                min_samples_leaf=2,
                min_samples_split=2
            ),

            'KNN (SMOTE)': KNeighborsClassifier(
                n_neighbors=15,
                weights='distance',
                metric='euclidean',
                p=1
            ),

            'SVM (SMOTE)': SVC(
                kernel='linear',        # common choice, can be 'linear' too
                C=1.0,               # regularization strength
                gamma='scale',       # auto scaling of kernel coefficient
                probability=True,    # enables predict_proba for ROC/AUC
                random_state=42
            )
        },

        # =========================================================
        # 2) MODELS USING CLASS WEIGHT 'balanced' (NO SMOTE)
        # =========================================================
        'supervised_weighted': {
            'Random Forest (Weighted)': RandomForestClassifier(
                n_estimators=200,
                random_state=42,
                max_depth=n_features,
                n_jobs=-1,
                min_samples_leaf=2,
                min_samples_split=2,
                class_weight='balanced'
            ),
            'XGBoost (Weighted)': XGBClassifier(
                n_estimators=200,
                random_state=42,
                max_depth=n_features,
                eval_metric='logloss',
                # for XGBoost "class_weight" doesn't exist;
                scale_pos_weight= 246/18
            ),
            'Logistic Regression (Weighted)': LogisticRegression(
                C=0.1,
                random_state=42,
                max_iter=1000,
                solver='liblinear',
                class_weight='balanced'
            ),
            'Decision Tree (Weighted)': DecisionTreeClassifier(
                random_state=42,
                max_depth=n_features,
                min_samples_leaf=2,
                min_samples_split=2,
                class_weight='balanced'
            ),
            'KNN (Imbalanced)': KNeighborsClassifier(
                n_neighbors=7,
                weights='distance',
                metric='euclidean',
                p=1
            ),
            'SVM (Weighted)': SVC(
                kernel='linear',        # common choice, can be 'linear' too
                C=1.0,               # regularization strength
                gamma='scale',       # auto scaling of kernel coefficient
                probability=True,    # enables predict_proba for ROC/AUC
                random_state=42,
                class_weight='balanced'
            )
        }
    }

    # Wrap into the structure expected by cv_metrics_table
    model_dicts = {}
    for group_name, group_models in models.items():
        tmp = {}
        for name, model in group_models.items():
            tmp[name] = {
                'model': model,
                'type': group_name
            }
        model_dicts[group_name] = tmp

    return model_dicts


In [ ]:
def train_all_models(X_train, y_train, model_group='supervised_smote'):
    """
    Train a group of models on the *already prepared* training data
    (e.g., after SMOTE / ADASYN / KMeansSMOTE, or even original).

    Parameters
    ----------
    X_train : array-like or DataFrame
        Training features (can already be oversampled and scaled).
    y_train : array-like or Series
        Training labels (aligned with X_train).
    model_group : str
        Which group from get_all_models() to use, e.g.:
        - 'supervised'        : if you trained on original data
        - 'supervised_smote'  : if you conceptually group these as SMOTE-based models
        - any other key you defined in get_all_models()

    Returns
    -------
    trained_models : dict
        { model_name: {'model': estimator, 'type': model_group, 'trained': True} }
    """

    models = get_all_models(X_train)          # your existing function
    if model_group not in models:
        raise ValueError(
            f"Model group '{model_group}' not found in get_all_models(). "
            f"Available groups: {list(models.keys())}"
        )

    model_dict = models[model_group]
    trained_models = {}

    print("\n" + "="*60)
    print(f"TRAINING MODELS ({model_group}) ON PROVIDED DATA")
    print("="*60)
    print(f"  - X_train shape: {X_train.shape}")
    print(f"  - y_train distribution: "
          f"0={sum(np.array(y_train)==0)}, 1={sum(np.array(y_train)==1)}")

    for name, model in model_dict.items():
        print(f"  - Training {name}...")
        model.fit(X_train, y_train)
        trained_models[name] = {
            'model': model,
            'type': model_group,
            'trained': True
        }

    print("\nAll models trained successfully!")
    return trained_models


### CROSS VALIDATION METHOD


In [ ]:
from imblearn.over_sampling import SMOTE, BorderlineSMOTE, ADASYN, KMeansSMOTE
from imblearn.combine import SMOTETomek, SMOTEENN


def get_oversampler(name="SMOTE", sampling_ratio=None, random_state=42):
    """
    Return an oversampler object from imbalanced-learn, based on a string name.

    Parameters
    ----------
    name : str
        One of:
        - "SMOTE"
        - "BorderlineSMOTE"
        - "ADASYN"
        - "KMeans-SMOTE"
        - "SMOTE-Tomek"
        - "SMOTE-ENN"

    sampling_ratio : float or dict or 'auto' or None
        Passed to sampling_strategy.
        Example: 0.5 -> minority = 50% of majority class.

    random_state : int
        Random seed for reproducibility.

    Returns
    -------
    oversampler : object with fit_resample(X, y)
    """
    if sampling_ratio is None:
        sampling_strategy = 'auto'
    else:
        sampling_strategy = sampling_ratio

    name = name.lower()

    if name == "smote":
        return SMOTE(sampling_strategy=sampling_strategy, random_state=random_state)

    elif name == "borderlinesmote":
        return BorderlineSMOTE(sampling_strategy=sampling_strategy, random_state=random_state)

    elif name == "adasyn":
        return ADASYN(sampling_strategy=sampling_strategy, random_state=random_state)

    elif name in ("kmeans-smote", "kmeanssmote"):
        return KMeansSMOTE(sampling_strategy=sampling_strategy, random_state=random_state)

    elif name in ("smote-tomek", "smotetomek"):
        return SMOTETomek(sampling_strategy=sampling_strategy, random_state=random_state)

    elif name in ("smote-enn", "smoteenn"):
        return SMOTEENN(sampling_strategy=sampling_strategy, random_state=random_state)

    else:
        raise ValueError(
            f"Unknown oversampler name '{name}'. "
            "Use one of: 'SMOTE', 'BorderlineSMOTE', 'ADASYN', "
            "'KMeans-SMOTE', 'SMOTE-Tomek', 'SMOTE-ENN'."
        )


In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, precision_recall_curve,
    auc
)
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.base import clone
import pandas as pd
import numpy as np


def cv_metrics_table(
    trained_models,
    X,
    y,
    use_smote_in_cv=True,
    oversampler_name="SMOTE",
    sampling_ratio=None,
    include_types=('supervised_smote',),
    combination_id=None,
    feature_names=None,
):
    """
    Cross-validated metrics table with per-model threshold tuning
    based on out-of-fold (OOF) predicted probabilities.

    Extra:
    - 'Combination #' column
    - 'Features' column (comma-separated list)
    - 'TNR' column (specificity) in addition to TPR.
    """

    X = np.asarray(X)
    y = np.asarray(y)

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    rows = []

    # build a pretty string for the Features column
    if feature_names is not None:
        features_str = ", ".join(feature_names)
    else:
        features_str = None

    for name, entry in trained_models.items():
        base_model = entry['model']
        mtype      = entry.get('type', None)

        if mtype not in include_types:
            continue

        # ---------------------------------------------------------
        # 1. Build estimator for CV
        # ---------------------------------------------------------
        if use_smote_in_cv:
            oversampler = get_oversampler(
                name=oversampler_name,
                sampling_ratio=sampling_ratio,
                random_state=42
            )

            estimator = ImbPipeline([
                ('oversampler', oversampler),
                ('clf', clone(base_model))
            ])
        else:
            estimator = clone(base_model)

        # OOF predicted probabilities for the positive class
        y_pred_proba = cross_val_predict(
            estimator, X, y,
            cv=cv, method='predict_proba'
        )[:, 1]

        # ---------------------------------------------------------
        # 2. THRESHOLD TUNING (on OOF probabilities)
        # ---------------------------------------------------------
        precision_arr, recall_arr, thresholds = precision_recall_curve(y, y_pred_proba)

        # F1 for each threshold; thresholds has len = len(precision_arr) - 1
        f1_scores = 2 * precision_arr * recall_arr / (precision_arr + recall_arr + 1e-8)
        best_idx = np.argmax(f1_scores[:-1])  # last element has no corresponding threshold
        # best_threshold = thresholds[best_idx]   # keep tuned threshold
        best_threshold = 0.5
        # Convert probs to labels using tuned threshold
        y_pred = (y_pred_proba >= best_threshold).astype(int)

        # ---------------------------------------------------------
        # 3. METRICS WITH TUNED THRESHOLD
        # ---------------------------------------------------------
        tn, fp, fn, tp = confusion_matrix(y, y_pred).ravel()

        precision = precision_score(y, y_pred, zero_division=0)
        recall    = recall_score(y, y_pred, zero_division=0)
        f1        = f1_score(y, y_pred, zero_division=0)
        rocauc    = roc_auc_score(y, y_pred_proba)

        # PR-AUC: area under Precision–Recall curve
        pr_auc    = auc(recall_arr, precision_arr)

        # TPR (Recall) and TNR (Specificity)
        tpr = recall
        tnr = tn / (tn + fp) if (tn + fp) > 0 else 0.0

        rows.append({
            "Combination #": combination_id,
            "Features": features_str,

            "Model": name,
            "Type": mtype,

            "Precision": precision,
            "Recall": recall,
            "F1-Score": f1,
            "ROC-AUC": rocauc,
            "PR-AUC": pr_auc,

            "TP": tp,
            "FP": fp,
            "FN": fn,
            "TN": tn,
            "TPR": tpr,
            "TNR": tnr,

            "BestThreshold": best_threshold,
            "Oversampler": oversampler_name if use_smote_in_cv else "None",
            "SamplingRatio": sampling_ratio,
        })

    # Enforce a nice column order similar to your example
    desired_order = [
        "Combination #", "Features", "Model", "Type",
        "Precision", "Recall", "F1-Score", "ROC-AUC", "PR-AUC",
        "TP", "FP", "FN", "TN", "TPR", "TNR",
        "BestThreshold", "Oversampler", "SamplingRatio"
    ]
    df = pd.DataFrame(rows)
    existing_cols = [c for c in desired_order if c in df.columns]
    df = df[existing_cols].sort_values(by="Combination #", ascending=True)

    return df


### Test Metrics Table

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix
)

def test_metrics_table(trained_models, cv_summary, X_test, y_test):
    """
    Evaluate tuned models on a TEST set using the thresholds
    previously selected via CV on the TRAIN set.

    Parameters
    ----------
    trained_models : dict
        Same structure you used before: {name: {'model': ..., 'type': ...}}.
        These models are already fitted on the TRAIN data.
    cv_summary : pd.DataFrame
        Output of cv_metrics_table on the TRAIN set, containing
        at least columns ['Model', 'BestThreshold'].
    X_test, y_test : array-like
        Held-out test data (no CV here).

    Returns
    -------
    pd.DataFrame
        Metrics on the test set for each model, using its CV-tuned threshold.
    """
    X_test = np.asarray(X_test)
    y_test = np.asarray(y_test)

    rows = []

    for name, entry in trained_models.items():
        base_model = entry['model']
        mtype      = entry['type']

        if mtype not in ["supervised", "supervised_smote"]:
            # skip anomaly / unsupervised etc.
            continue

        # 1) Get the best threshold from TRAIN CV summary
        row = cv_summary.loc[cv_summary["Model"] == name]
        if row.empty:
            # model was not in cv_summary_1 (or got filtered out)
            continue
        best_threshold = row["BestThreshold"].values[0]

        # 2) Predict probabilities on the TEST set
        #    (for SMOTE pipelines, you should have refit the pipeline on TRAIN
        #     before building `trained_models`).
        y_proba = base_model.predict_proba(X_test)[:, 1]

        # 3) Apply threshold
        y_pred = (y_proba >= best_threshold).astype(int)

        # 4) Compute metrics
        tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

        precision = precision_score(y_test, y_pred, zero_division=0)
        recall    = recall_score(y_test, y_pred, zero_division=0)
        f1        = f1_score(y_test, y_pred, zero_division=0)
        rocauc    = roc_auc_score(y_test, y_proba)

        rows.append({
            "Model": name,
            "Type": mtype,
            "BestThreshold": best_threshold,
            "Precision": precision,
            "Recall": recall,
            "F1-Score": f1,
            "ROC-AUC": rocauc,
            "True Positives": tp,
            "False Positives": fp,
            "False Negatives": fn,
            "True Negatives": tn,
        })

    return pd.DataFrame(rows).sort_values(by="F1-Score", ascending=False)


In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix
)

def test_metrics_from_final(final_models, final_thresholds, X_test, y_test):
    """
    Evaluate already-fitted final models on TEST set, using
    per-model thresholds that were tuned on the TRAIN set.

    Parameters
    ----------
    final_models : dict
        {model_name: fitted_estimator}
        (e.g. final_models_S1, final_models_S2, ...)
    final_thresholds : dict
        {model_name: best_threshold_from_train}
        (e.g. final_thresholds_S1, ...)
    X_test, y_test : array-like
        Held-out test set.

    Returns
    -------
    pd.DataFrame
        Test metrics for each model.
    """
    X_test = np.asarray(X_test)
    y_test = np.asarray(y_test)

    rows = []

    for name, estimator in final_models.items():
        if name not in final_thresholds:
            continue

        thr = final_thresholds[name]

        # probabilities on TEST
        proba = estimator.predict_proba(X_test)[:, 1]
        y_pred = (proba >= thr).astype(int)

        tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()
        prec = precision_score(y_test, y_pred, zero_division=0)
        rec  = recall_score(y_test, y_pred, zero_division=0)
        f1   = f1_score(y_test, y_pred, zero_division=0)
        roc  = roc_auc_score(y_test, proba)

        rows.append({
            "Model": name,
            "BestThreshold_train": thr,
            "Precision_Test": prec,
            "Recall_Test": rec,
            "F1_Test": f1,
            "ROC-AUC_Test": roc,
            "TP_Test": tp,
            "FP_Test": fp,
            "FN_Test": fn,
            "TN_Test": tn,
        })

    return pd.DataFrame(rows).sort_values(
        by="F1_Test", ascending=False
    ).reset_index(drop=True)


### Model Hyperparameter Tuning

#### RANDOM FOREST 

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold

def tune_random_forest_resampled(X_res, y_res, sampler_name=""):
    """
    Hyperparameter tuning for Random Forest on already-resampled data.
    No class_weight, no SMOTE inside CV.

    Parameters
    ----------
    X_res, y_res : resampled training data (e.g. from SMOTE, ADASYN)
    sampler_name : optional string, just for printing ("SMOTE", "ADASYN", ...)

    Returns
    -------
    best_estimator_, best_params_, best_score_
    """

    rf_base = RandomForestClassifier(
        random_state=42,
        max_depth=X_res.shape[1],
        n_jobs=-1
    )

    param_grid = {
        'n_estimators':      [100, 200, 500],
        'max_depth':         [2, 3, 4, 5],
        'min_samples_split': [2, 3, 5],
        'min_samples_leaf':  [1, 3, 5],
        'max_features':      ['sqrt', 'log2', 0.5],
    }

    cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

    print(f"\n===== Tuning Random Forest on {sampler_name or 'resampled'} data =====")
    grid = GridSearchCV(
        estimator=rf_base,
        param_grid=param_grid,
        scoring='recall',
        cv=cv,
        n_jobs=-1,
        verbose=1
    )
    grid.fit(X_res, y_res)

    print("Best RF params:", grid.best_params_)
    print("Best RF CV F1 :", grid.best_score_)

    return grid.best_estimator_, grid.best_params_, grid.best_score_, grid.cv_results_


#### XGBOOST

In [ ]:
from xgboost import XGBClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold

def tune_xgboost_resampled(X_res, y_res, sampler_name=""):
    """
    Hyperparameter tuning for XGBoost on already-resampled data.
    Do NOT use scale_pos_weight here (class is already rebalanced).
    """

    # Base model (scale_pos_weight will be tuned)
    xgb_base = XGBClassifier(
        eval_metric="logloss",
        random_state=42,
        n_estimators=200,
        max_depth=X_res.shape[1],
        # early_stopping_rounds=20,
        use_label_encoder=False,  # optional depending on xgboost version
    )

    # compute approximate neg/pos ratio:
    n_pos = (y_res == 1).sum()
    n_neg = (y_res == 0).sum()
    ratio = n_neg / max(n_pos, 1)

    param_grid = {
        "n_estimators": [100, 200],
        "learning_rate": [0.01, 0.05, 0.1],
        # 'max_depth':         [2, 3, 4],
        "min_child_weight": [3, 4, 5],
        "subsample": [0.6, 0.8],
        "colsample_bytree": [0.6, 0.8],
        "gamma": [0.0, 0.1, 0.5],
        # tune around the empirical imbalance ratio
        "scale_pos_weight": [0.5 * ratio, ratio],
    }

    cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

    print(f"\n===== Tuning XGBoost on {sampler_name or 'resampled'} data =====")
    grid = GridSearchCV(
        estimator=xgb_base,
        param_grid=param_grid,
        scoring='recall',
        cv=cv,
        n_jobs=-1,
        verbose=1
    )
    grid.fit(X_res, y_res)

    print("Best XGB params:", grid.best_params_)
    print("Best XGB CV F1 :", grid.best_score_)

    return grid.best_estimator_, grid.best_params_, grid.best_score_, grid.cv_results_


#### LOGISTIC REGRESSION

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold

def tune_logistic_regression_resampled(X_res, y_res, sampler_name=""):
    """
    Hyperparameter tuning for Logistic Regression on already-resampled data.
    No class_weight, since oversampling already handled imbalance.
    """

    lr_base = LogisticRegression(
        solver='liblinear',   # supports l1 and l2
        max_iter=1000,
        random_state=42
    )

    param_grid = {
        'C': [0.001, 0.01, 0.1, 1],
        'penalty': ['l1'],
    }

    cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

    print(f"\n===== Tuning Logistic Regression on {sampler_name or 'resampled'} data =====")
    grid = GridSearchCV(
        estimator=lr_base,
        param_grid=param_grid,
        scoring='f1',
        cv=cv,
        n_jobs=-1,
        verbose=1
    )
    grid.fit(X_res, y_res)

    print("Best LR params:", grid.best_params_)
    print("Best LR CV F1 :", grid.best_score_)

    return grid.best_estimator_, grid.best_params_, grid.best_score_, grid.cv_results_


#### KNN TUNING

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold

def tune_knn_resampled(X_res, y_res, sampler_name=""):
    """
    Hyperparameter tuning for KNN on already-resampled data.
    """

    knn_base = KNeighborsClassifier()

    param_grid = {
        'n_neighbors': [5, 6, 7, 9, 10, 12, 15],
        'weights': 'distance',
        'metric': ['euclidean','manhattan','minkowski'],
        'p': [1]
    }

    cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

    print(f"\n===== Tuning KNN on {sampler_name or 'resampled'} data =====")
    grid = GridSearchCV(
        estimator=knn_base,
        param_grid=param_grid,
        scoring='f1',
        cv=cv,
        n_jobs=-1,
        verbose=1
    )
    grid.fit(X_res, y_res)

    print("Best KNN params:", grid.best_params_)
    print("Best KNN CV F1 :", grid.best_score_)

    return grid.best_estimator_, grid.best_params_, grid.best_score_, grid.cv_results_


#### DECISION TREE TUNING

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold

def tune_decision_tree_resampled(X_res, y_res, sampler_name=""):
    """
    Hyperparameter tuning for Decision Tree on already-resampled data.
    No class_weight here.
    """

    dt_base = DecisionTreeClassifier(
        class_weight="balanced", 
        max_depth=X_res.shape[1], 
        random_state=42
    )

    param_grid = {
        "criterion": ["gini", "entropy", "log_loss"],
        "min_samples_split": [2, 3, 4, 5, 10],
        "min_samples_leaf": [1, 2, 3, 4],
        "max_features": ["sqrt", "log2", None],
    }

    cv = StratifiedKFold(n_splits=10, shuffle=True, random_state=42)

    grid = GridSearchCV(
        estimator=dt_base,
        param_grid=param_grid,
        scoring="f1",
        cv=cv,
        n_jobs=-1,
        verbose=1,
    )

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

    print(f"\n===== Tuning Decision Tree on {sampler_name or 'resampled'} data =====")
    grid = GridSearchCV(
        estimator=dt_base,
        param_grid=param_grid,
        scoring='f1',
        cv=cv,
        n_jobs=-1,
        verbose=1
    )
    grid.fit(X_res, y_res)

    print("Best DT params:", grid.best_params_)
    print("Best DT CV F1 :", grid.best_score_)

    return grid.best_estimator_, grid.best_params_, grid.best_score_, grid.cv_results_


# Start of Main Algorithm

## Data preparation

In [ ]:
# ======================================================================
# STEP 1: LOAD AND PREPARE DATA (REFERENCE + MULTI-KIT MONORAIL)
# ======================================================================

def load_data(model_path, monorail_paths):
    """
    Load and prepare reference (model) data and Monorail data (one or more kits),
    align common columns, and return a single combined DataFrame.

    Parameters
    ----------
    model_path : str
        Path to model.csv (reference experimental campaign).
    monorail_paths : str or list of str
        Path or list of paths to Monorail TestBrakefinal_data_kitXX.csv files.

    Returns
    -------
    df_base : pandas.DataFrame
        Combined DataFrame with:
        - aligned common columns between reference and Monorail,
        - binary label (0/1) where available,
        - 'Source' column (kit ID or 0 for reference),
        - 'DataSource' column (0 = reference, 1 = Monorail).
    """

    # --------------------------------------------------------------
    # Helper: load and clean ONE Monorail file
    # --------------------------------------------------------------
    def load_Monorail(filepath: str) -> pd.DataFrame:
        df = pd.read_csv(filepath)

        # Keep only standard braking
        if 'Non_Standard_Braking' in df.columns:
            df = df[df['Non_Standard_Braking'] == 0]

        # Extract numeric kit ID from filename, e.g. "TestBrakefinal_data_kit06.csv" -> 6
        match = re.search(r'Dati(\d+)', os.path.basename(filepath))
        source = int(match.group(1)) if match else -1
        df['Source'] = source

        # Convert "xx sec" string columns to float seconds where possible
        for col in df.select_dtypes(include='object'):
            try:
                df[col] = df[col].str.replace(' sec', '', regex=False).astype(float)
            except (AttributeError, ValueError):
                # AttributeError if column is not string-like; ValueError if some values cannot be cast
                continue

        return df

    # --------------------------------------------------------------
    # 1) REFERENCE DATA: load, label, aggregate
    # --------------------------------------------------------------
    df_reference = pd.read_csv(model_path)
    df_reference['Malfunction'] = df_reference['Malfunction'].astype(str)

    # Binary label from malfunction code
    leakage_codes = ['C', 'D', 'E', 'F', 'G']
    df_reference['LeakageLabel'] = np.where(
        df_reference['Malfunction'].isin(leakage_codes),
        'Combined leakage',
        'Healthy'
    )

    # Add Source = 0 for reference campaign
    df_reference['Source'] = 0

    # Aggregate delay and efficiency columns
    delay_eff_map = {
        'Total_timing_delay':      ['Brake_timing_delay_exp',      'Release_timing_delay_exp'],
        'Total_energy_delay':      ['Brake_energy_delay_exp',      'Release_energy_delay_exp'],
        'Total_power_delay':       ['Brake_power_delay_exp',       'Release_power_delay_exp'],
        'Total_power_efficiency':  ['Brake_power_efficiency_exp',  'Release_power_efficiency_exp'],
        'Total_energy_efficiency': ['Brake_energy_effiency_exp',   'Release_energy_efficiency_exp']
    }

    for new_col, (c1, c2) in delay_eff_map.items():
        # If any of these columns are missing in some version of model.csv, guard with .get
        if c1 in df_reference.columns and c2 in df_reference.columns:
            df_reference[new_col] = df_reference[c1] + df_reference[c2]

    # Drop original per-phase columns (only those that actually exist)
    cols_to_drop = [c for pair in delay_eff_map.values() for c in pair if c in df_reference.columns]
    df_reference.drop(columns=cols_to_drop, inplace=True, errors='ignore')

    # Rename to your canonical names
    rename_map = {
        'Release_start_pressure_delay_exp': 'Release_start_pressure_delay',
        'Buildup_end_pressure_delay_exp':  'Buildup_end_pressure_delay',
        'Weight':                          'WV_MeanPressure',
        'Brake_action':                    'EmergencyBrake_action'
    }
    df_reference.rename(columns=rename_map, inplace=True)

    # --------------------------------------------------------------
    # 2) MONORAIL DATA: load one or more kit files
    # --------------------------------------------------------------
    if isinstance(monorail_paths, str):
        monorail_paths = [monorail_paths]

    dfs_mono = [load_Monorail(fp) for fp in monorail_paths]
    df_data = pd.concat(dfs_mono, ignore_index=True)

    # --------------------------------------------------------------
    # 3) ALIGN STRUCTURES AND COMBINE
    # --------------------------------------------------------------
    # Ensure 'Source' is integer in both
    df_reference['Source'] = df_reference['Source'].astype(int)
    df_data['Source']      = df_data['Source'].astype(int)

    # Columns common to BOTH datasets
    common_cols = df_reference.columns.intersection(df_data.columns).tolist()

    # Subsets with only common columns + a DataSource flag
    df_reference_subset = df_reference[common_cols].copy()
    df_reference_subset['DataSource'] = 0  # 0 = reference campaign

    df_data_subset = df_data[common_cols].copy()
    df_data_subset['DataSource'] = 1       # 1 = Monorail (real-time) data

    # Stack reference + Monorail
    df_combined = pd.concat([df_reference_subset, df_data_subset], ignore_index=True)

    # Encode final label column (will be NaN for Monorail if it has no LeakageLabel)
    if 'LeakageLabel' in df_combined.columns:
        df_combined.rename(columns={'LeakageLabel': 'label'}, inplace=True)
        df_combined['label'] = df_combined['label'].map({'Healthy': 0, 'Combined leakage': 1})

    # Convert any remaining "xx sec" string columns to float (esp. from model.csv)
    for col in df_combined.select_dtypes(include='object'):
        try:
            df_combined[col] = df_combined[col].str.replace(' sec', '', regex=False).astype(float)
        except (AttributeError, ValueError):
            continue
    df_combined["WV_bin"] = df_combined["WV_MeanPressure"].apply(
    lambda p: np.nan if pd.isna(p) else (0 if p < 2 else (2 if p > 3 else 1))
    )
    df_base = df_combined.copy()
    return df_base

model_path = 'model.csv'
monorail_paths = [
    'TestBrakefinal_data_Dati01.csv',
    'TestBrakefinal_data_Dati06.csv',
    'TestBrakefinal_data_Dati27.csv'
]

df = load_data(model_path, monorail_paths)

print(df.shape)
print(df['DataSource'].value_counts(dropna=False))
print(df['label'].value_counts(dropna=False))  # will include NaN for unlabeled Monorail

In [ ]:
df.head()

## PREPROCESS DATA FOR ML

In [ ]:
# Select the Loading Condition that is within 2 and 3 bar
# Instead of directly filtering, lets create WV_bin column for <2 bar, 2 to 3 bar, and >3 bar 
# based on WV_MeanPressure

from sklearn.model_selection import train_test_split

def preprocess_data(df, features, test_size):
    """
    Preprocess data: create WV_bin column, filter by bin==1, select features,
    split into train/test, and separate healthy samples for training.
    
    Parameters:
    - df: pandas DataFrame containing the data
    - features: list of feature names to use (e.g., ['Total_power_efficiency', 'FlowRate'])
    - test_size: proportion of the dataset to include in the test split
    
    Returns:
    - X_train, X_test, y_train, y_test, X_train_healthy
    """
    
    # Create WV_bin column
    df = df.copy()
    # Filter by WV_bin == 1 (pressure between 2 and 3)
    df_filt = df[df["WV_bin"] == 1].copy()
    
    # Validate feature selection
    missing = [f for f in features if f not in df_filt.columns]
    if missing:
        raise ValueError(f"The following features are not in the dataframe: {missing}")
    
    # Select features and labels
    X = df_filt[features]
    y = df_filt['label']
    
    # Split data
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=test_size, stratify=y, random_state=42
    )
    
    # Extract healthy training samples
    X_train_healthy = X_train[y_train == 0]
    
    print(f"Total samples: {len(df)}")
    print(f"Filtered samples (WV_bin==1): {len(df_filt)}")
    print(f"Training samples: {len(X_train)} (Healthy: {sum(y_train==0)}, Leakage: {sum(y_train==1)})")
    print(f"Training samples (healthy only): {len(X_train_healthy)}")
    print(f"Test samples: {len(X_test)} (Healthy: {sum(y_test==0)}, Leakage: {sum(y_test==1)})")
    
    return X_train, X_test, y_train, y_test, X_train_healthy

In [ ]:
from sklearn.impute import SimpleImputer

def scale_features(X_train, X_test, X_train_healthy):
    """
    Impute missing values by median, then standardize features.
    Returns imputed+scaled arrays, plus fitted scaler and imputer.
    """
    # 1) Median imputation (fit only on training set)
    imputer = SimpleImputer(strategy='median')
    X_train_imp = imputer.fit_transform(X_train)
    X_test_imp = imputer.transform(X_test)
    X_train_healthy_imp = imputer.transform(X_train_healthy)

    # 2) Standardization (fit only on imputed training set)
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train_imp)
    X_test_scaled = scaler.transform(X_test_imp)
    X_train_healthy_scaled = scaler.transform(X_train_healthy_imp)

    return X_train_scaled, X_test_scaled, X_train_healthy_scaled, scaler, imputer

## Data Exploration Continues

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# ------------------------------------------------------------
# 1. FIXED jitter vector (same for both plots)
# ------------------------------------------------------------
rng = np.random.default_rng(42)   # reproducibility
df['y_jitter'] = rng.normal(0, 0.02, size=len(df))

# ------------------------------------------------------------
# 2. Palettes
# ------------------------------------------------------------
palette_source = sns.color_palette("colorblind", n_colors=len(df['Source'].unique()))
label_palette = {
    0: (0.2, 0.4, 0.9, 0.35),   # RGBA → semi-transparent blue
    1: (1.0, 0.55, 0.0, 1.0),   # solid orange
}

# df_filtered = df[(df['EmergencyBrake_action'] == 1) & (df['WV_bin'] == 1)].copy()
# df_filtered = df[(df['DataSource'] == 0) & (df['WV_bin'] == 1)].copy()
df_filtered = df[(df['WV_bin'] == 1)].copy()
# df_filtered = df.copy()
# palette_label  = sns.color_palette("Set1", n_colors=len(df['label'].unique()))

# Map categories to colors
source_codes = df_filtered['Source'].astype('category').cat.codes
label_codes  = df_filtered['label'].astype('category').cat.codes

colors_source = [palette_source[c] for c in source_codes]
colors_label  = [label_palette[c]  for c in label_codes]

# ------------------------------------------------------------
# 3. Plot side-by-side layout
# ------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)

# ---------------- Left: coloured by Source ----------------
axes[0].scatter(
    df_filtered['Total_power_efficiency'],
    df_filtered['y_jitter'],
    c=colors_source,
    alpha=0.7
)
axes[0].set_title("Total Power Efficiency distribution by Kit Source")
axes[0].set_xlabel("Total Power Efficiency")
axes[0].set_yticks([])

# Source legend
sources = df_filtered['Source'].unique()
for i, s in enumerate(sources):
    axes[0].scatter([], [], c=[palette_source[i]], label=str(s))
axes[0].legend(title="Kit Source", loc="upper right")
axes[0].set_xlim(0, 11)
# ---------------- Right: coloured by label ----------------
axes[1].scatter(
    df_filtered['Total_power_efficiency'],
    df_filtered['y_jitter'],
    c=colors_label,
    alpha=0.7
)
axes[1].set_title("Total Power Efficiency distribution by Label")
axes[1].set_xlabel("Total Power Efficiency")
axes[1].set_yticks([])

# Label legend
labels = df_filtered['label'].unique()
for i, lab in enumerate(labels):
    axes[1].scatter([], [], c=[label_palette[i]], label=str(lab))
axes[1].legend(title="Leakage Label", loc="upper right")
axes[1].set_xlim(0, 11)
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# ------------------------------------------------------------
# 1. FIXED jitter vector (same for both plots)
# ------------------------------------------------------------
rng = np.random.default_rng(42)   # reproducibility
df['y_jitter'] = rng.normal(0, 0.02, size=len(df))
# df_filtered = df[(df['Max_pressure_pipe'] < 0.8) & (df['WV_bin'] == 1)].copy()
# df_filtered = df[(df['EmergencyBrake_action'] == 0) & (df['WV_bin'] == 1)].copy()
df_filtered = df[
    (df['DataSource'] == 0) & 
    (df['WV_bin'].isin([1]))
].copy()
# df_filtered = df[(df['DataSource'] == 0) & (df['WV_bin'] == 1)].copy()
# df_filtered = df[(df['WV_bin'] == 1)].copy()
# ------------------------------------------------------------
# 2. Palettes
# ------------------------------------------------------------
palette_source = sns.color_palette("colorblind", n_colors=len(df['Source'].unique()))
label_palette = {
    0: (0.2, 0.4, 0.9, 0.35),   # RGBA → semi-transparent blue
    1: (1.0, 0.55, 0.0, 1.0),   # solid orange
}


# palette_label  = sns.color_palette("Set1", n_colors=len(df['label'].unique()))

# Map categories to colors
source_codes = df_filtered['Source'].astype('category').cat.codes
label_codes  = df_filtered['label'].astype('category').cat.codes

colors_source = [palette_source[c] for c in source_codes]
colors_label  = [label_palette[c]  for c in label_codes]

# ------------------------------------------------------------
# 3. Plot side-by-side layout
# ------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)

# ---------------- Left: coloured by Source ----------------
axes[0].scatter(
    df_filtered['Buildup_end_pressure_delay'],
    df_filtered['y_jitter'],
    c=colors_source,
    alpha=0.7
)
axes[0].set_title("Total Power Delay distribution by Kit Source")
axes[0].set_xlabel("Total Power Delay")
axes[0].set_yticks([])

# Source legend
sources = df_filtered['Source'].unique()
for i, s in enumerate(sources):
    axes[0].scatter([], [], c=[palette_source[i]], label=str(s))
axes[0].legend(title="Kit Source", loc="upper right")

# ---------------- Right: coloured by label ----------------
axes[1].scatter(
    df_filtered['Buildup_end_pressure_delay'],
    df_filtered['y_jitter'],
    c=colors_label,
    alpha=0.7
)
axes[1].set_title("Total Power Delay distribution by Label")
axes[1].set_xlabel("Total Power Delay")
axes[1].set_yticks([])

# Label legend
labels = df_filtered['label'].unique()
for i, lab in enumerate(labels):
    axes[1].scatter([], [], c=[label_palette[i]], label=str(lab))
axes[1].legend(title="Leakage Label", loc="upper right")
axes[1].set_xlim(-1, 2)
plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt

def plot_efficiency_vs_features(df, features, base_width=7, base_height=5, x_limits=None):
    """
    Plot Total_power_efficiency against each selected feature.
    Creates N separate figures (one per feature).
    
    Parameters:
    -----------
    df : pandas.DataFrame
        Data containing 'Total_power_efficiency', 'label', and selected features.
    features : list of str
        List of feature column names to plot against Total_power_efficiency.
    base_width : int, optional
        Width of each figure (default=7).
    base_height : int, optional
        Height of each figure (default=5).
    x_limits : tuple (min, max), optional
        Limits for the X-axis (Total_power_efficiency).
    """
    # Masks for labels
    mask_0 = df['label'] == 0
    mask_1 = df['label'] == 1
    
    for feature in features:
        plt.figure(figsize=(base_width, base_height))
        
        plt.scatter(
            df.loc[mask_0, 'Total_energy_efficiency'],
            df.loc[mask_0, feature],
            color='blue', alpha=0.4, label='0'
        )
        plt.scatter(
            df.loc[mask_1, 'Total_energy_efficiency'],
            df.loc[mask_1, feature],
            color='red', alpha=0.7, label='1'
        )
        
        plt.xlabel("Total_energy_efficiency")
        plt.ylabel(feature)
        plt.title(f"Total_energy_efficiency vs {feature}")
        plt.legend(title='label', bbox_to_anchor=(1.05, 1), loc='upper left')
        
        # Apply X-axis limits if provided
        if x_limits is not None:
            plt.xlim(x_limits)
        
        plt.tight_layout()
        plt.show()

# Example usage:
df_filtered = df[df['WV_bin'] == 1]

plot_features = ["Total_power_delay", "WV_MeanPressure","Total_energy_efficiency","Std_delay_exp"]

# Limit X-axis between 0 and 100
plot_efficiency_vs_features(df_filtered, plot_features, x_limits=(0, 40))


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import seaborn as sns

# ------------------------------------------------------------
# 1. FIXED jitter vector (same for both plots)
# ------------------------------------------------------------
rng = np.random.default_rng(42)   # reproducibility
df_filt = df[df["WV_bin"] == 1].copy()
df_filt['y_jitter'] = rng.normal(0, 0.02, size=len(df_filt))

# ------------------------------------------------------------
# 2. Palettes
# ------------------------------------------------------------
palette_source = sns.color_palette("colorblind", n_colors=len(df_filt['Source'].unique()))
label_palette = {
    0: (0.2, 0.4, 0.9, 0.35),   # RGBA → semi-transparent blue
    1: (1.0, 0.55, 0.0, 1.0),   # solid orange
}


# palette_label  = sns.color_palette("Set1", n_colors=len(df['label'].unique()))

# Map categories to colors
source_codes = df_filt['Source'].astype('category').cat.codes
label_codes  = df_filt['label'].astype('category').cat.codes

colors_source = [palette_source[c] for c in source_codes]
colors_label  = [label_palette[c]  for c in label_codes]

# ------------------------------------------------------------
# 3. Plot side-by-side layout
# ------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 4), sharey=True)

# ---------------- Left: coloured by Source ----------------
axes[0].scatter(
    df_filt['Buildup_end_pressure_delay'],
    df_filt['y_jitter'],
    c=colors_source,
    alpha=0.7
)
axes[0].set_title("Total Power Delay distribution by Kit Source")
axes[0].set_xlabel("Total Power Delay")
axes[0].set_yticks([])

# Source legend
sources = df_filt['Source'].unique()
for i, s in enumerate(sources):
    axes[0].scatter([], [], c=[palette_source[i]], label=str(s))
axes[0].legend(title="Kit Source", loc="upper right")

# ---------------- Right: coloured by label ----------------
axes[1].scatter(
    df_filt['Buildup_end_pressure_delay'],
    df_filt['y_jitter'],
    c=colors_label,
    alpha=0.7
)
axes[1].set_title("Total Power Delay distribution by Label")
axes[1].set_xlabel("Total Power Delay")
axes[1].set_yticks([])

# Label legend
labels = df_filt['label'].unique()
for i, lab in enumerate(labels):
    axes[1].scatter([], [], c=[label_palette[i]], label=str(lab))
axes[1].legend(title="Leakage Label", loc="upper right")

plt.tight_layout()
plt.show()


## Algorithm START

In [ ]:
import pandas as pd
from itertools import combinations
from sklearn.preprocessing import StandardScaler  # if not already imported

# -------------------------------------------------------------------
# GIVEN
# -------------------------------------------------------------------
selected_features = [
    'Total_power_efficiency',
    'Total_power_delay',
    'Std_delay_exp',
    'Total_energy_delay',
    'Total_energy_efficiency'
]

# Your existing functions must already be defined:
# - preprocess_data(df, feature_list, test_size=0.2)
# - scale_features(X_train, X_test, X_train_healthy)
#
# And you already have `df` available as your main dataset.

# -------------------------------------------------------------------
# BUILD ALL COMBINATIONS (2-, 3-, and 4-feature subsets)
# -------------------------------------------------------------------
feature_combinations = []
for r in [2, 3, 4, 5]:
    for combination in combinations(selected_features, r):
        feature_combinations.append(list(combination))  # convert tuple -> list for convenience

# -------------------------------------------------------------------
# FOR EACH COMBINATION: PREPROCESS + SCALE + STORE IN DICT
# -------------------------------------------------------------------
combination_data = {}      # main dictionary with all data per combination
summary_rows = []    # to build a summary DataFrame

for combination_idx, feat_list in enumerate(feature_combinations, start=1):
    # 1) Preprocess for this specific subset of features
    X_train, X_test, y_train, y_test, X_train_healthy = preprocess_data(
        df,
        feat_list,
        test_size=0.2
    )

    # Align y_train_healthy with X_train_healthy index
    y_train_healthy = y_train.loc[X_train_healthy.index]

    # 2) Impute + scale for this subset
    X_train_scaled, X_test_scaled, X_train_healthy_scaled, scaler, imputer = scale_features(
        X_train,
        X_test,
        X_train_healthy
    )

    # 3) Store everything in a dictionary under this combination index
    combination_data[combination_idx] = {
        "features": feat_list,

        "X_train": X_train,
        "X_test": X_test,
        "y_train": y_train,
        "y_test": y_test,
        "X_train_healthy": X_train_healthy,
        "y_train_healthy": y_train_healthy,

        "X_train_scaled": X_train_scaled,
        "X_test_scaled": X_test_scaled,
        "X_train_healthy_scaled": X_train_healthy_scaled,

        "scaler": scaler,
        "imputer": imputer,
    }

    # 4) Add row for the summary table
    summary_rows.append({
        "Combination #": combination_idx,
        "Features": ", ".join(feat_list),
        "n_features": len(feat_list),
    })

# -------------------------------------------------------------------
# SUMMARY TABLE OF COMBINATIONS (LIKE YOUR FEATURE COLUMN)
# -------------------------------------------------------------------
combination_summary = pd.DataFrame(summary_rows).sort_values("Combination #")

# Example: look at it
# print(combination_summary)


In [ ]:
c7 = combination_data[7]
X_train_scaled_7 = c7["X_train_scaled"]
y_train_7        = c7["y_train"]
features_7       = c7["features"]


### SMOTE

In [ ]:
all_cv_results = []
cv_results_per_combination = {}

for combination_id, info in combination_data.items():
    # 1) Extract scaled training data for this combination
    X_train_scaled = info["X_train_scaled"]
    y_train = info["y_train"]
    feat_list = info["features"]

    # 2) Build model dictionary for this number of features
    n_features = X_train_scaled.shape[1]
    models_all = get_all_models(n_features)
    trained_models_proto = models_all['supervised_smote']

    cv_results = cv_metrics_table(
        trained_models=trained_models_proto,
        X=X_train_scaled,
        y=y_train,
        use_smote_in_cv=True,
        sampling_ratio=0.3,
        oversampler_name="ADASYN",
        include_types=('supervised_smote',),
        combination_id=combination_id,
        feature_names=feat_list
    )


    # 4) Collect this combination's table
    all_cv_results.append(cv_results)
    cv_results_per_combination[combination_id] = cv_results

# 5) Final Training-Set table over ALL combinations + ALL models
training_cv_table = pd.concat(all_cv_results, ignore_index=True)

# Optional: sort by Combination then Model
training_cv_table = training_cv_table.sort_values(
    by=["Combination #", "Model"]
).reset_index(drop=True)

In [ ]:
training_cv_table.head()

Try the UNTUNNED MODEL on Test Data

In [ ]:
from sklearn.base import clone

def fit_final_models_with_oversampling(
    proto_models,
    X_train,
    y_train,
    oversampler_name="ADASYN",
    sampling_ratio=None
):
    """
    Fit final models on the full TRAIN set with oversampling.
    proto_models: dict {name: {'model': estimator, 'type': str}}
    Returns: dict in the same structure but with fitted estimators.
    """
    # Reuse the same helper you used in cv_metrics_table
    oversampler = get_oversampler(
        name=oversampler_name,
        sampling_ratio=sampling_ratio,
        random_state=42
    )

    X_res, y_res = oversampler.fit_resample(X_train, y_train)

    final_models = {}
    for name, entry in proto_models.items():
        base_model = clone(entry['model'])
        base_model.fit(X_res, y_res)

        final_models[name] = {
            'model': base_model,
            'type': entry.get('type', 'supervised_smote')
        }

    return final_models


In [ ]:
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, precision_recall_curve, auc
)
import numpy as np
import pandas as pd

def test_metrics_table_for_combination(
    trained_models,
    cv_summary,
    X_test,
    y_test,
    combination_id=None,
    feature_names=None
):
    """
    Evaluate final (fitted) models on TEST set using thresholds
    previously selected via CV (cv_summary).

    trained_models : dict {name: {'model': fitted_estimator, 'type': str}}
    cv_summary     : DataFrame from cv_metrics_table for this combination
    X_test, y_test : test data (already scaled)
    """
    rows = []

    if feature_names is not None:
        features_str = ", ".join(feature_names)
    else:
        features_str = None

    for name, entry in trained_models.items():
        model = entry['model']
        mtype = entry.get('type', None)

        # 1) Find the threshold for this model from CV summary
        row = cv_summary[cv_summary["Model"] == name]
        if row.empty:
            # if for some reason it's not in cv_summary, skip
            continue
        best_threshold = row["BestThreshold"].iloc[0]

        # 2) Predict probabilities on TEST set
        y_proba = model.predict_proba(X_test)[:, 1]
        y_pred  = (y_proba >= best_threshold).astype(int)

        # 3) Metrics
        tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

        precision = precision_score(y_test, y_pred, zero_division=0)
        recall    = recall_score(y_test, y_pred, zero_division=0)
        f1        = f1_score(y_test, y_pred, zero_division=0)
        rocauc    = roc_auc_score(y_test, y_proba)

        precision_arr, recall_arr, _ = precision_recall_curve(y_test, y_proba)
        pr_auc = auc(recall_arr, precision_arr)

        tpr = recall
        tnr = tn / (tn + fp) if (tn + fp) > 0 else 0.0

        rows.append({
            "Combination #": combination_id,
            "Features": features_str,
            "Model": name,
            "Type": mtype,

            "Precision": precision,
            "Recall": recall,
            "F1-Score": f1,
            "ROC-AUC": rocauc,
            "PR-AUC": pr_auc,

            "TP": tp,
            "FP": fp,
            "FN": fn,
            "TN": tn,
            "TPR": tpr,
            "TNR": tnr,

            "BestThreshold": best_threshold
        })

    columns_order = [
        "Combination #", "Features", "Model", "Type",
        "Precision", "Recall", "F1-Score", "ROC-AUC", "PR-AUC",
        "TP", "FP", "FN", "TN", "TPR", "TNR",
        "BestThreshold"
    ]
    df = pd.DataFrame(rows)
    df = df[columns_order]
    return df


In [ ]:
all_test_results = []

for combination_id, info in combination_data.items():
    # Training data for this combination
    X_train_scaled = info["X_train_scaled"]
    y_train        = info["y_train"]

    # Test data for this combination
    X_test_scaled  = info["X_test_scaled"]
    y_test         = info["y_test"]

    feat_list      = info["features"]

    # 1) Prototype models
    n_features = X_train_scaled.shape[1]
    models_all = get_all_models(n_features)
    trained_models_smote = models_all['supervised_smote']

    # 2) Fit final models on full TRAIN with oversampling
    final_models = fit_final_models_with_oversampling(
        trained_models_smote,
        X_train_scaled,
        y_train,
        oversampler_name="ADASYN",   # same as in CV
        sampling_ratio=0.3          # or your 0.3 if you want
    )

    # 3) Get the CV summary for this combination
    cv_summary = cv_results_per_combination[combination_id]

    # 4) Evaluate on TEST using CV thresholds
    test_results = test_metrics_table_for_combination(
        trained_models=final_models,
        cv_summary=cv_summary,
        X_test=X_test_scaled,
        y_test=y_test,
        combination_id=combination_id,
        feature_names=feat_list
    )

    all_test_results.append(test_results)

# Final TEST table (same structure as training table but from test set)
test_table = pd.concat(all_test_results, ignore_index=True)
test_table = test_table.sort_values(by=["Combination #", "Model"]).reset_index(drop=True)


### BALANCED CLASS WEIGHT (WEIGHTED MODEL)

In [ ]:
all_cv_weighted = []
cv_weighted_per_combination = {}

for combination_id, info in combination_data.items():
    X_train_scaled = info["X_train_scaled"]
    y_train        = info["y_train"]
    feat_list      = info["features"]

    n_features = X_train_scaled.shape[1]
    models_all = get_all_models(n_features)
    trained_models_weighted = models_all['supervised_weighted']

    cv_results_weighted = cv_metrics_table(
        trained_models=trained_models_weighted,
        X=X_train_scaled,
        y=y_train,
        use_smote_in_cv=False,
        oversampler_name=None,
        include_types=('supervised_weighted',),
        combination_id=combination_id,
        feature_names=feat_list
    )

    cv_weighted_per_combination[combination_id] = cv_results_weighted
    all_cv_weighted.append(cv_results_weighted)

training_cv_table_weighted = pd.concat(all_cv_weighted, ignore_index=True)

#### Simple model fit

In [ ]:
from sklearn.base import clone

def fit_final_models_weighted(proto_models, X_train, y_train):
    """
    Fit final class-weighted models directly on full TRAIN set
    (no oversampling).
    """
    final_models = {}
    for name, entry in proto_models.items():
        base_model = clone(entry['model'])
        base_model.fit(X_train, y_train)

        final_models[name] = {
            'model': base_model,
            'type': entry.get('type', 'supervised_weighted')
        }

    return final_models


Test Loop for Weighted Model

In [ ]:
all_test_weighted = []

for combination_id, info in combination_data.items():
    X_train_scaled = info["X_train_scaled"]
    y_train        = info["y_train"]
    X_test_scaled  = info["X_test_scaled"]
    y_test         = info["y_test"]
    feat_list      = info["features"]

    n_features = X_train_scaled.shape[1]
    models_all = get_all_models(n_features)
    proto_models_weighted = models_all['supervised_weighted']

    # 1) Fit final weighted models on the full TRAIN set
    final_models_weighted = fit_final_models_weighted(
        proto_models_weighted, X_train_scaled, y_train
    )

    # 2) Get the CV summary (for thresholds) for this combination
    cv_summary_weighted = cv_weighted_per_combination[combination_id]

    # 3) Evaluate on TEST using those thresholds
    test_results_weighted = test_metrics_table_for_combination(
        trained_models=final_models_weighted,
        cv_summary=cv_summary_weighted,
        X_test=X_test_scaled,
        y_test=y_test,
        combination_id=combination_id,
        feature_names=feat_list
    )

    all_test_weighted.append(test_results_weighted)

test_table_weighted = pd.concat(all_test_weighted, ignore_index=True)
test_table_weighted = test_table_weighted.sort_values(
    by=["Combination #", "Model"]
).reset_index(drop=True)


### Learning the Oversample effect

In [ ]:
from sklearn.decomposition import PCA
import matplotlib.pyplot as plt
from imblearn.over_sampling import ADASYN
import numpy as np

def plot_pca_for_combination(combination_id, combination_data, sampling_strategy=0.3):
    """
    Generate PCA plot before and after ADASYN for a specific feature combination.
    For visualization ONLY (NOT used in training).
    """
    
    # --------------------------------------------------
    # 1. GET THE TRAINING DATA FOR THIS COMBINATION
    # --------------------------------------------------
    X_train_scaled = combination_data[combination_id]["X_train_scaled"]
    y_train        = combination_data[combination_id]["y_train"].values  # ensure numpy array

    # --------------------------------------------------
    # 2. Fit PCA only on the ORIGINAL data
    # --------------------------------------------------
    pca = PCA(n_components=2)
    X_orig_2d = pca.fit_transform(X_train_scaled)

    # --------------------------------------------------
    # 3. Apply ADASYN ONLY FOR VISUALIZATION
    # --------------------------------------------------
    adasyn = ADASYN(sampling_strategy=sampling_strategy, random_state=42)
    X_sm, y_sm = adasyn.fit_resample(X_train_scaled, y_train)

    # Transform ADASYN data using the SAME PCA
    X_sm_2d = pca.transform(X_sm)

    # --------------------------------------------------
    # 4. Plot ORIGINAL
    # --------------------------------------------------
    plt.figure(figsize=(6, 5))
    plt.scatter(X_orig_2d[y_train == 0, 0], X_orig_2d[y_train == 0, 1],
                alpha=0.5, s=15, label="Healthy (orig)")
    plt.scatter(X_orig_2d[y_train == 1, 0], X_orig_2d[y_train == 1, 1],
                alpha=0.6, s=30, label="Faulty (orig)")
    plt.axhline(0, color='gray', linestyle='--', linewidth=0.6)
    plt.axvline(0, color='gray', linestyle='--', linewidth=0.6)
    plt.title(f"PCA – ORIGINAL (Combination {combination_id})")
    plt.xlabel("PC1"); plt.ylabel("PC2")
    plt.legend(); plt.tight_layout()
    plt.show()

    # --------------------------------------------------
    # 5. Plot AFTER ADASYN
    # --------------------------------------------------
    plt.figure(figsize=(6, 5))
    plt.scatter(X_sm_2d[y_sm == 0, 0], X_sm_2d[y_sm == 0, 1],
                alpha=0.7, s=15, label="Healthy (ADASYN)")
    plt.scatter(X_sm_2d[y_sm == 1, 0], X_sm_2d[y_sm == 1, 1],
                alpha=0.3, s=30, label="Faulty (ADASYN)")
    plt.axhline(0, color='gray', linestyle='--', linewidth=0.6)
    plt.axvline(0, color='gray', linestyle='--', linewidth=0.6)
    plt.title(f"PCA – AFTER ADASYN (Combination {combination_id})")
    plt.xlabel("PC1"); plt.ylabel("PC2")
    plt.legend(); plt.tight_layout()
    plt.show()

plot_pca_for_combination(combination_id=1, combination_data=combination_data)


In [ ]:
for cid in combination_data.keys():
    plot_pca_for_combination(cid, combination_data)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.decomposition import PCA

# -------------------------------------------------
# 1. Find the combination that uses ALL selected features
# -------------------------------------------------
full_combination_id = None
for cid, info in combination_data.items():
    feats = info["features"]
    if set(feats) == set(selected_features):
        full_combination_id = cid
        break

if full_combination_id is None:
    raise ValueError("No combination in combination_data uses exactly selected_features.")

# Get the scaled training data for that combination
X_train_scaled_full = combination_data[full_combination_id]["X_train_scaled"]

# -------------------------------------------------
# 2. Fit PCA on the ORIGINAL data with all components
# -------------------------------------------------
n_feats = len(selected_features)
pca_full = PCA(n_components=n_feats)
pca_full.fit(X_train_scaled_full)

# -------------------------------------------------
# 3. Scree plot
# -------------------------------------------------
explained_var = pca_full.explained_variance_ratio_ * 100

plt.figure(figsize=(6, 4))
plt.bar(range(1, len(explained_var) + 1), explained_var)
plt.title("Scree Plot – Explained Variance by Principal Components")
plt.xlabel("Principal Component")
plt.ylabel("Explained Variance [%]")
plt.xticks(range(1, len(explained_var) + 1))
plt.grid(True, linestyle='--', alpha=0.6)

# Cumulative variance
plt.plot(
    range(1, len(explained_var) + 1),
    np.cumsum(explained_var),
    'o--',
    label="Cumulative"
)
plt.legend()
plt.tight_layout()
plt.show()

print("Explained variance ratio (%):", explained_var)
print("Cumulative variance (%):    ", np.cumsum(explained_var))

# -------------------------------------------------
# 4. Loadings table (feature contributions to PCs)
# -------------------------------------------------
pc_names = [f"PC{i+1}" for i in range(pca_full.n_components_)]
loadings = pd.DataFrame(
    pca_full.components_.T,
    columns=pc_names,
    index=selected_features
)

print(loadings)


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from sklearn.neighbors import KNeighborsClassifier
from imblearn.over_sampling import ADASYN
from matplotlib.colors import ListedColormap

def plot_knn_pca_decision_boundary_for_combination(
    combination_id,
    combination_data,
    knn_best=None,          # optional: best tuned KNN, else we set defaults
    sampling_strategy=0.3
):
    """
    Plot KNN decision boundary in 2D PCA space for a given feature combination,
    using ADASYN-resampled data ONLY for visualization.
    """
    # -------------------------------
    # 1. Get this combination's train data
    # -------------------------------
    X_train_scaled = combination_data[combination_id]["X_train_scaled"]
    y_train        = combination_data[combination_id]["y_train"].values

    # -------------------------------
    # 2. PCA (2D) fitted on ORIGINAL
    # -------------------------------
    pca = PCA(n_components=2)
    X_orig_2d = pca.fit_transform(X_train_scaled)

    # -------------------------------
    # 3. ADASYN for visualization
    # -------------------------------
    adasyn = ADASYN(sampling_strategy=sampling_strategy, random_state=42)
    X_sm, y_sm = adasyn.fit_resample(X_train_scaled, y_train)

    # Project ADASYN data into same PCA space
    X_sm_2d = pca.transform(X_sm)

    # -------------------------------
    # 4. Train KNN in 2D PCA space
    # -------------------------------
    if knn_best is not None:
        best_params = knn_best.get_params()
        k_for_plot  = best_params.get('n_neighbors', 5)
        knn_pca = KNeighborsClassifier(
            n_neighbors=k_for_plot,
            weights=best_params.get('weights', 'uniform'),
            metric=best_params.get('metric', 'minkowski'),
            p=best_params.get('p', 2)
        )
    else:
        k_for_plot = 5
        knn_pca = KNeighborsClassifier(
            n_neighbors=k_for_plot,
            weights='uniform',
            metric='minkowski',
            p=2
        )

    knn_pca.fit(X_sm_2d, y_sm)

    # -------------------------------
    # 5. Create grid over PCA space
    # -------------------------------
    x_min, x_max = X_sm_2d[:, 0].min() - 1, X_sm_2d[:, 0].max() + 1
    y_min, y_max = X_sm_2d[:, 1].min() - 1, X_sm_2d[:, 1].max() + 1

    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 300),
        np.linspace(y_min, y_max, 300)
    )

    grid_points = np.c_[xx.ravel(), yy.ravel()]
    Z = knn_pca.predict(grid_points)
    Z = Z.reshape(xx.shape)

    # -------------------------------
    # 6. Plot decision regions + data
    # -------------------------------
    plt.figure(figsize=(6, 5))

    cmap_background = ListedColormap(['#AAAAFF', '#FFAAAA'])
    cmap_points     = ListedColormap(['#0000FF', '#FF0000'])

    # decision regions
    plt.contourf(xx, yy, Z, alpha=0.3, cmap=cmap_background)

    # ADASYN samples in PCA space
    plt.scatter(
        X_sm_2d[y_sm == 0, 0], X_sm_2d[y_sm == 0, 1],
        c='b', alpha=0.6, s=15, label="Healthy (ADASYN)"
    )
    plt.scatter(
        X_sm_2d[y_sm == 1, 0], X_sm_2d[y_sm == 1, 1],
        c='r', alpha=0.2, s=30, label="Leakage (ADASYN)"
    )

    plt.title(f"KNN decision boundary in PCA space (Combination {combination_id}, k = {k_for_plot})")
    plt.xlabel("PC1")
    plt.ylabel("PC2")
    plt.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
# Example: use the full-feature combination we found before
plot_knn_pca_decision_boundary_for_combination(
    combination_id=2,
    combination_data=combination_data,
    knn_best=None  # or your tuned KNN for that combination
)


## Try to train the model using PCA feature space

In [ ]:
from sklearn.decomposition import PCA
import numpy as np

def add_pca_to_data_library(combination_data, n_components=None, var_threshold=None):
    """
    For each combination in combination_data:
      - Fit PCA on X_train_scaled
      - Transform X_train_scaled and X_test_scaled into PC space
      - Store X_train_pca, X_test_pca, and the fitted PCA object

    Parameters
    ----------
    combination_data : dict
        Your existing combination_data[combination_id] structure.
    n_components : int or None
        If int -> use this many PCs.
        If None and var_threshold is None -> use all features.
    var_threshold : float or None
        If given (e.g. 0.95), choose the minimal number of PCs
        that explain at least this proportion of variance.
    """
    for cid, info in combination_data.items():
        X_train_scaled = info["X_train_scaled"]
        X_test_scaled  = info["X_test_scaled"]

        # Fit PCA on TRAIN only
        pca = PCA()
        pca.fit(X_train_scaled)

        # Decide how many components
        if var_threshold is not None:
            cum_var = np.cumsum(pca.explained_variance_ratio_)
            k = np.searchsorted(cum_var, var_threshold) + 1
        elif n_components is not None:
            k = n_components
        else:
            k = X_train_scaled.shape[1]  # all PCs

        # Re-fit with chosen k
        pca_k = PCA(n_components=k)
        X_train_pca = pca_k.fit_transform(X_train_scaled)
        X_test_pca  = pca_k.transform(X_test_scaled)

        # Store in combination_data
        info["X_train_pca"] = X_train_pca
        info["X_test_pca"]  = X_test_pca
        info["pca"]         = pca_k
        info["n_pca"]       = k

# e.g. keep enough PCs to for 2 components
add_pca_to_data_library(combination_data, n_components=2)

# or: force exactly 2 PCs for all combinations:
# add_pca_to_data_library(combination_data, n_components=2)


### Train the PCA feature space with SMOTE

In [ ]:
all_cv_pca_smote = []
cv_pca_smote_per_combination = {}

for combination_id, info in combination_data.items():
    X_train_pca = info["X_train_pca"]
    y_train     = info["y_train"]
    feat_list   = info["features"]
    n_pca       = info["n_pca"]

    # models, but now with n_features = n_pca
    models_all = get_all_models(n_pca)
    trained_models_smote = models_all['supervised_smote']

    cv_results_pca = cv_metrics_table(
        trained_models=trained_models_smote,
        X=X_train_pca,
        y=y_train,
        use_smote_in_cv=True,         # still oversampling, but now in PC space
        oversampler_name="ADASYN",
        include_types=('supervised_smote',),
        combination_id=combination_id,
        feature_names=[f"PC{i+1}" for i in range(n_pca)]  # or keep original names if you prefer
    )

    cv_pca_smote_per_combination[combination_id] = cv_results_pca
    all_cv_pca_smote.append(cv_results_pca)

training_cv_table_pca_smote = pd.concat(all_cv_pca_smote, ignore_index=True)
# Save to CSV
training_cv_table_pca_smote.to_csv("training_cv_table_pca_smote.csv", index=False)


### Train the PCA feature space with class balanced

In [ ]:
all_cv_pca_weighted = []
cv_pca_weighted_per_combination = {}

for combination_id, info in combination_data.items():
    X_train_pca = info["X_train_pca"]
    y_train     = info["y_train"]
    n_pca       = info["n_pca"]
    feat_list   = info["features"]

    models_all = get_all_models(n_pca)
    trained_models_weighted = models_all['supervised_weighted']

    cv_results_pca_w = cv_metrics_table(
        trained_models=trained_models_weighted,
        X=X_train_pca,
        y=y_train,
        use_smote_in_cv=False,          # no oversampling
        oversampler_name=None,
        include_types=('supervised_weighted',),
        combination_id=combination_id,
        feature_names=[f"PC{i+1}" for i in range(n_pca)]
    )

    cv_pca_weighted_per_combination[combination_id] = cv_results_pca_w
    all_cv_pca_weighted.append(cv_results_pca_w)

training_cv_table_pca_weighted = pd.concat(all_cv_pca_weighted, ignore_index=True)
# Save to CSV
training_cv_table_pca_weighted.to_csv("training_cv_table_pca_weighted.csv", index=False)

### Test the PCA trained

In [ ]:
all_test_pca_weighted = []

for combination_id, info in combination_data.items():
    X_train_pca = info["X_train_pca"]
    y_train     = info["y_train"]
    X_test_pca  = info["X_test_pca"]
    y_test      = info["y_test"]
    n_pca       = info["n_pca"]

    models_all = get_all_models(n_pca)
    proto_models_weighted = models_all['supervised_weighted']

    # Fit final models on full TRAIN (no oversampling)
    final_models_weighted_pca = fit_final_models_weighted(
        proto_models_weighted,
        X_train_pca,
        y_train
    )

    # Use the WEIGHTED-PCA CV summary, not SMOTE
    cv_summary = cv_pca_weighted_per_combination[combination_id]

    test_results_pca_weighted = test_metrics_table_for_combination(
        trained_models=final_models_weighted_pca,
        cv_summary=cv_summary,
        X_test=X_test_pca,
        y_test=y_test,
        combination_id=combination_id,
        feature_names=[f"PC{i+1}" for i in range(n_pca)]
    )

    all_test_pca_weighted.append(test_results_pca_weighted)

test_table_pca_weighted = pd.concat(all_test_pca_weighted, ignore_index=True)
test_table_pca_weighted.to_csv("test_table_pca_weighted.csv", index=False)


In [ ]:
all_test_pca_smote = []

for combination_id, info in combination_data.items():
    X_train_pca = info["X_train_pca"]
    y_train     = info["y_train"]
    X_test_pca  = info["X_test_pca"]
    y_test      = info["y_test"]
    n_pca       = info["n_pca"]

    models_all = get_all_models(n_pca)
    proto_models_smote = models_all['supervised_smote']

    final_models_smote_pca = fit_final_models_with_oversampling(
        proto_models_smote,
        X_train_pca,
        y_train,
        oversampler_name="ADASYN",   # same as in CV
        sampling_ratio=0.3          # or your 0.3 if you want
    )

    cv_summary = cv_pca_smote_per_combination[combination_id]

    test_results_pca_smote = test_metrics_table_for_combination(
        trained_models=final_models_smote_pca,
        cv_summary=cv_summary,
        X_test=X_test_pca,
        y_test=y_test,
        combination_id=combination_id,
        feature_names=[f"PC{i+1}" for i in range(n_pca)]
    )

    all_test_pca_smote.append(test_results_pca_smote)

test_table_pca_smote = pd.concat(all_test_pca_smote, ignore_index=True)
# Save to CSV
test_table_pca_smote.to_csv("test_table_pca_smote.csv", index=False)

## Plot KNN Decision Boundary on Real Space (2 Features)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.base import clone
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler
from matplotlib.colors import ListedColormap
from imblearn.over_sampling import ADASYN

def plot_decision_boundary_real_space(
    combination_id,
    combination_data,
    estimator=None,
    model_name=None,
    train_with_oversampling=True,
    sampling_strategy=0.3,
    tuned_estimator=None,
    point_set="train",
    xlim=None, 
    ylim=None
):
    """
    Plot decision boundary in REAL FEATURE SPACE for a given 2-feature
    combination from combination_data, using an arbitrary sklearn classifier.

    The model is ALWAYS trained on TRAIN data (optionally with ADASYN),
    and you can choose whether to plot TRAIN or TEST points.

    Parameters
    ----------
    combination_id : int
        Key of the combination in combination_data.

    combination_data : dict
        combination_data[combination_id] must contain at least:
            - "features": list of feature names
            - "X_train":  DataFrame (unscaled training features)
            - "y_train":  Series/array of training labels
            - "X_test":   DataFrame (unscaled test features)
            - "y_test":   Series/array of test labels

    estimator : sklearn estimator or None
        Unfitted sklearn classifier (KNN, Tree, LogisticRegression, etc.).
        If None, a default KNN is used.

    model_name : str or None
        Name for plot title. If None, inferred from estimator class.

    train_with_oversampling : bool, default=True
        If True, apply ADASYN to TRAIN data for fitting the model.
        TEST data is never oversampled.

    sampling_strategy : float, default=0.3
        ADASYN sampling_strategy (minority fraction) for training.

    tuned_estimator : sklearn estimator or None
        If provided, its hyperparameters are reused to build the 2D model
        (it will still be refit on the 2D data inside this function).

    point_set : {"train", "test"}, default="train"
        Which points to overlay on the decision boundary:
        - "train" → show training samples
        - "test"  → show test samples
    """

    info = combination_data[combination_id]
    feat_list = info["features"]

    # We need exactly 2 features to draw a 2D boundary
    if len(feat_list) != 2:
        raise ValueError(
            f"Combination {combination_id} has {len(feat_list)} features, "
            "but this plot requires exactly 2."
        )

    feat_x, feat_y = feat_list[0], feat_list[1]

    # -------------------------------------------------
    # 1. Extract 2D TRAIN and TEST in REAL FEATURE SPACE
    # -------------------------------------------------
    X_train_2d = info["X_train"][[feat_x, feat_y]].values
    y_train    = np.asarray(info["y_train"])

    X_test_2d  = info["X_test"][[feat_x, feat_y]].values
    y_test     = np.asarray(info["y_test"])

    # -------------------------------------------------
    # 2. Choose training data (optionally ADASYN)
    # -------------------------------------------------
    if train_with_oversampling:
        smote = KMeansSMOTE(sampling_strategy=sampling_strategy, random_state=42)
        X_fit, y_fit = smote.fit_resample(X_train_2d, y_train)
    else:
        X_fit, y_fit = X_train_2d, y_train

    # -------------------------------------------------
    # 3. Choose which points to plot (train or test)
    # -------------------------------------------------
    if point_set == "train":
        X_vis = X_train_2d
        y_vis = y_train
        point_label = "TRAIN"
    elif point_set == "test":
        X_vis = X_test_2d
        y_vis = y_test
        point_label = "TEST"
    else:
        raise ValueError("point_set must be 'train' or 'test'.")

    # -------------------------------------------------
    # 4. Scale inputs for the model (axes remain in real units)
    # -------------------------------------------------
    scaler = StandardScaler()
    X_fit_scaled = scaler.fit_transform(X_fit)
    X_vis_scaled = scaler.transform(X_vis)

    # -------------------------------------------------
    # 5. Build and fit the classifier
    # -------------------------------------------------
    if estimator is not None:
        base_est = clone(estimator)
    elif tuned_estimator is not None:
        base_est = tuned_estimator.__class__(**tuned_estimator.get_params())
    else:
        # Default: simple KNN
        base_est = KNeighborsClassifier(
            n_neighbors=5,
            weights='uniform',
            metric='minkowski',
            p=2
        )

    base_est.fit(X_fit_scaled, y_fit)

    if model_name is None:
        model_name = base_est.__class__.__name__

    # -------------------------------------------------
    # 6. Create a grid in REAL FEATURE SPACE
    # -------------------------------------------------
    x_range = np.ptp(X_vis[:, 0])
    y_range = np.ptp(X_vis[:, 1])

    x_min = X_vis[:, 0].min() - 0.1 * abs(x_range)
    x_max = X_vis[:, 0].max() + 0.1 * abs(x_range)

    y_min = X_vis[:, 1].min() - 0.1 * abs(y_range)
    y_max = X_vis[:, 1].max() + 0.1 * abs(y_range)

    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 300),
        np.linspace(y_min, y_max, 300)
    )

    grid_real   = np.c_[xx.ravel(), yy.ravel()]
    grid_scaled = scaler.transform(grid_real)
    Z = base_est.predict(grid_scaled)
    Z = Z.reshape(xx.shape)

    # -------------------------------------------------
    # 7. Plot decision regions + chosen points
    # -------------------------------------------------
    plt.figure(figsize=(6, 5))

    cmap_background = ListedColormap(['#AAAAFF', '#FFAAAA'])

    # decision regions
    plt.contourf(xx, yy, Z, alpha=0.3, cmap=cmap_background)

    # selected point set (train or test) in real feature space
    plt.scatter(
        X_vis[y_vis == 0, 0], X_vis[y_vis == 0, 1],
        alpha=0.6, s=15, label=f"Healthy ({point_label})"
    )
    plt.scatter(
        X_vis[y_vis == 1, 0], X_vis[y_vis == 1, 1],
        alpha=0.8, s=30, label=f"Faulty ({point_label})"
    )

    plt.title(f"{model_name} decision boundary in real feature space\n"
              f"Combination {combination_id} – points: {point_label}")
    plt.xlabel(feat_x)
    plt.ylabel(feat_y)
    plt.xlim(xlim)
    plt.ylim(ylim)
    plt.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
plot_decision_boundary_real_space(
    combination_id=2,
    combination_data=combination_data,
    train_with_oversampling=True,
    sampling_strategy=0.3,
    point_set="train",
    xlim=(0, 10), 
    ylim=(-0.8, 0.6)
    )


In [ ]:
plot_decision_boundary_real_space(
    combination_id=2,
    combination_data=combination_data,
    train_with_oversampling=False,   # ADASYN on TRAIN
    sampling_strategy=0.3,
    point_set="train",
    xlim=(0, 10), 
    ylim=(-0.8, 0.6)
    )

In [ ]:
plot_decision_boundary_real_space(
    combination_id=2,
    combination_data=combination_data,
    train_with_oversampling=True,
    sampling_strategy=0.3,
    point_set="test",
    xlim=(1, 9), 
    ylim=(-0.6, 0.5)
)


In [ ]:
import pandas as pd
from sklearn.decomposition import PCA
import numpy as np

def compute_pca2_variance(combination_data, start=1, end=10):
    """
    Compute PCA(2) explained variance for each combination_id in range.
    Returns a DataFrame:
        Combination # | Feature Count | PC1 Var | PC2 Var | PC1+PC2
    """
    rows = []

    for cid in range(start, end+1):
        info = combination_data[cid]

        X = info["X_train_scaled"]   # scaled features
        feats = info["features"]
        k = len(feats)

        pca2 = PCA(n_components=2)
        pca2.fit(X)

        pc1, pc2 = pca2.explained_variance_ratio_

        rows.append({
            "Combination #": cid,
            "Features": ", ".join(feats),
            "Feature Count": k,
            "PC1 (%)": pc1 * 100,
            "PC2 (%)": pc2 * 100,
            "PC1+PC2 (%)": (pc1 + pc2) * 100
        })

    return pd.DataFrame(rows)

pca2_table = compute_pca2_variance(combination_data, start=1, end=10)
pca2_table


In [ ]:
pca3_table = compute_pca2_variance(combination_data, start=11, end=20)
pca3_table

In [ ]:
def plot_pca2_components(pca2_table):
    plt.figure(figsize=(8,5))

    plt.bar(pca2_table["Combination #"], pca2_table["PC1 (%)"], label="PC1")
    plt.bar(pca2_table["Combination #"], pca2_table["PC2 (%)"], 
            bottom=pca2_table["PC1 (%)"], label="PC2")

    plt.xlabel("Combination #")
    plt.ylabel("Variance Explained (%)")
    plt.title("PCA(2) – Breakdown of PC1 and PC2 Across Combinations")
    plt.ylim(0, 100)
    plt.legend()
    plt.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()
plot_pca2_components(pca2_table)


In [ ]:
plot_pca2_components(pca3_table)

### Plot Logistic Regression Decision Boundary

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.base import clone
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from matplotlib.colors import ListedColormap
from imblearn.over_sampling import ADASYN

def plot_decision_boundary_logreg(
    combination_id,
    combination_data,
    estimator=None,
    model_name=None,
    train_with_oversampling=True,
    sampling_strategy=0.3,
    tuned_estimator=None,
    point_set="train",
    xlim=None, 
    ylim=None
):
    """
    Plot decision boundary in REAL FEATURE SPACE for a given 2-feature
    combination from combination_data, using Logistic Regression (default) or
    any sklearn classifier.
    """

    info = combination_data[combination_id]
    feat_list = info["features"]

    # Require exactly 2 features
    if len(feat_list) != 2:
        raise ValueError(
            f"Combination {combination_id} has {len(feat_list)} features, "
            "but this plot requires exactly 2."
        )

    feat_x, feat_y = feat_list[0], feat_list[1]

    # 1. Extract 2D TRAIN and TEST
    X_train_2d = info["X_train"][[feat_x, feat_y]].values
    y_train    = np.asarray(info["y_train"])

    X_test_2d  = info["X_test"][[feat_x, feat_y]].values
    y_test     = np.asarray(info["y_test"])

    # 2. Oversampling (optional)
    if train_with_oversampling:
        ada = ADASYN(sampling_strategy=sampling_strategy, random_state=42)
        X_fit, y_fit = ada.fit_resample(X_train_2d, y_train)
    else:
        X_fit, y_fit = X_train_2d, y_train

    # 3. Choose which points to plot
    if point_set == "train":
        X_vis, y_vis, point_label = X_train_2d, y_train, "TRAIN"
    elif point_set == "test":
        X_vis, y_vis, point_label = X_test_2d, y_test, "TEST"
    else:
        raise ValueError("point_set must be 'train' or 'test'.")

    # 4. Scale inputs for the model
    scaler = StandardScaler()
    X_fit_scaled = scaler.fit_transform(X_fit)
    X_vis_scaled = scaler.transform(X_vis)

    # 5. Build and fit classifier
    if estimator is not None:
        base_est = clone(estimator)
    elif tuned_estimator is not None:
        base_est = tuned_estimator.__class__(**tuned_estimator.get_params())
    else:
        # Default: Logistic Regression
        base_est = LogisticRegression(
            solver='lbfgs',
            max_iter=1000,
            random_state=42
        )

    base_est.fit(X_fit_scaled, y_fit)

    if model_name is None:
        model_name = base_est.__class__.__name__

    # 6. Create a grid in REAL FEATURE SPACE
    x_range = np.ptp(X_vis[:, 0])
    y_range = np.ptp(X_vis[:, 1])

    x_min = X_vis[:, 0].min() - 0.1 * abs(x_range)
    x_max = X_vis[:, 0].max() + 0.1 * abs(x_range)

    y_min = X_vis[:, 1].min() - 0.1 * abs(y_range)
    y_max = X_vis[:, 1].max() + 0.1 * abs(y_range)

    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 300),
        np.linspace(y_min, y_max, 300)
    )

    grid_real   = np.c_[xx.ravel(), yy.ravel()]
    grid_scaled = scaler.transform(grid_real)
    Z = base_est.predict(grid_scaled)
    Z = Z.reshape(xx.shape)

    # 7. Plot decision regions + chosen points
    plt.figure(figsize=(6, 5))
    cmap_background = ListedColormap(['#AAAAFF', '#FFAAAA'])

    plt.contourf(xx, yy, Z, alpha=0.3, cmap=cmap_background)

    plt.scatter(
        X_vis[y_vis == 0, 0], X_vis[y_vis == 0, 1],
        alpha=0.6, s=15, label=f"Healthy ({point_label})"
    )
    plt.scatter(
        X_vis[y_vis == 1, 0], X_vis[y_vis == 1, 1],
        alpha=0.8, s=30, label=f"Faulty ({point_label})"
    )

    plt.title(f"{model_name} decision boundary in real feature space\n"
              f"Combination {combination_id} – points: {point_label}")
    plt.xlabel(feat_x)
    plt.ylabel(feat_y)
    plt.xlim(xlim)
    plt.ylim(ylim)
    plt.legend()
    plt.tight_layout()
    plt.show()

In [ ]:
plot_decision_boundary_logreg(
    combination_id=2,
    combination_data=combination_data,
    train_with_oversampling=True,
    sampling_strategy=0.7,
    point_set="test",
    xlim=(1, 9), 
    ylim=(-0.6, 0.5)
)


### Plot Confusion Matrix

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_confusion_from_results(
    results_df,
    combination_id,
    model_name,
    title_prefix="Test set"
):
    """
    Plot confusion matrix for a given model and combination
    using a results table that contains TP, FP, FN, TN.

    Parameters
    ----------
    results_df : pd.DataFrame
        e.g. test_table_weighted or test_table_
        Must contain columns:
        ['Combination #', 'Model', 'TP', 'FP', 'FN', 'TN'].

    combination_id : int
        Value in the 'Combination #' column.

    model_name : str
        Exact string in the 'Model' column.

    title_prefix : str, optional
        Prefix for the plot title, e.g. 'Test set (Weighted)'.
    """
    # Filter the row
    row = results_df[
        (results_df["Combination #"] == combination_id) &
        (results_df["Model"] == model_name)
    ]

    if row.empty:
        raise ValueError(
            f"No row found for Combination #{combination_id} and Model '{model_name}'."
        )

    r = row.iloc[0]

    # Build confusion matrix in standard layout:
    # rows = true [0, 1], cols = predicted [0, 1]
    cm = np.array([
        [r["TN"], r["FP"]],
        [r["FN"], r["TP"]]
    ])

    classes = ["Healthy (0)", "Faulty (1)"]

    fig, ax = plt.subplots(figsize=(4.5, 4))
    im = ax.imshow(cm, interpolation="nearest", cmap=plt.cm.Blues)

    ax.figure.colorbar(im, ax=ax)
    ax.set(
        xticks=np.arange(cm.shape[1]),
        yticks=np.arange(cm.shape[0]),
        xticklabels=["Pred Healthy", "Pred Leakage"],
        yticklabels=["True Healthy", "True Leakage"],
        ylabel="True label",
        xlabel="Predicted label"
    )

    # Rotate x tick labels
    plt.setp(ax.get_xticklabels(), rotation=45, ha="right",
             rotation_mode="anchor")

    # Annotate each cell
    thresh = cm.max() / 2.0
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(
                j, i, int(cm[i, j]),
                ha="center", va="center",
                color="white" if cm[i, j] > thresh else "black"
            )

    title = f"{title_prefix} – {model_name}\nCombination #{combination_id}"
    ax.set_title(title)
    fig.tight_layout()
    plt.show()


In [ ]:
plot_confusion_from_results(
    test_table_weighted,
    combination_id=2,
    model_name="KNN (Imbalanced)",
    title_prefix="Test set (Weighted)"
)


In [ ]:
plot_confusion_from_results(
    test_table_weighted,
    combination_id=2,
    model_name="Decision Tree (Weighted)",
    title_prefix="Test set (Weighted)"
)

In [ ]:
plot_confusion_from_results(
    test_table_weighted,
    combination_id=15,
    model_name="Decision Tree (Weighted)",
    title_prefix="Test set (Weighted)"
)

In [ ]:
plot_confusion_from_results(
    test_table,
    combination_id=15,
    model_name="Logistic Regression (SMOTE)",
    title_prefix="Test set (SMOTE)"
)


In [ ]:
plot_confusion_from_results(
    test_table,
    combination_id=15,
    model_name="KNN (SMOTE)",
    title_prefix="Test set (SMOTE)"
)
